In [38]:
import time
from typing import Callable, Union, Union
import torch
import torch.nn.functional as F
from torch.optim import Optimizer, SGD
from torch.utils.data import DataLoader
from torch import Tensor
import argparse
import json
import tensorboard
import tensorboardX
import os
import argparse
import json
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim 
import nni
from nni.nas.nn.pytorch import ModelSpace, LayerChoice, MutableConv2d, MutableBatchNorm2d, MutableReLU
from pytorch_lightning import Trainer
from nni.nas.evaluator.pytorch import Lightning, ClassificationModule, Trainer
from nni.nas.experiment import NasExperiment
from nni.nas.space import model_context
from nni.nas.hub.pytorch import DARTS
from nni.nas.strategy import DARTS as DartsStrategy
from pytorch_lightning.loggers import TensorBoardLogger
from torch.utils.data import DataLoader
from torch.utils.data.sampler import SubsetRandomSampler
from torchvision import transforms
from torchvision.datasets import CIFAR10
from nni.nas.experiment import NasExperiment
from nni.nas.evaluator import FunctionalEvaluator
from nni.nas.evaluator import FunctionalEvaluator
import nni.nas.strategy as strategy
from torchvision import transforms
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader
#from ops import AvgPool,DilConv,SepConv
import genotypes
from pytorch_lightning.callbacks import ModelCheckpoint
torch.set_float32_matmul_precision('medium')
from tqdm import tqdm
from nni.nas.nn.pytorch import LayerChoice, ModelSpace,ValueChoice
from torch.utils.data import DataLoader, Dataset, SubsetRandomSampler
from pytorch_lightning import LightningModule, Trainer
from torchvision import datasets, transforms
from nni.nas.evaluator.pytorch import Classification
from nni.common.types import SCHEDULER
import nni
from nni.compression.quantization import QATQuantizer
from nni.compression.utils import TorchEvaluator
from torch.nn import utils
import torch.nn.utils as nn_utils
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
import time
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

# 1. Hyperparameters
BATCH_SIZE = 16
EPOCHS = 600
LEARNING_RATE = 0.0025
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [39]:
class PhotonicArch(torch.nn.Module):
    def __init__(self, drop_path_prob=0.0):
        super().__init__()
        
        self.drop_path_prob = drop_path_prob 
        #________________________________________________________________________________________________________________________
        #Layer 0
        self.layer0_conv = torch.nn.Conv2d(3, 8, kernel_size=3, padding=0, bias=False)
        self.layer0_bn = torch.nn.BatchNorm2d(8)
        #________________________________________________________________________________________________________________________
        #Layer 1
        self.layer1_avgpool= torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
        self.layer1_conv=torch.nn.Conv2d(8, 64, kernel_size=3, stride=1, padding=1)
        self.layer1_bn=torch.nn.BatchNorm2d(64, affine=True)
        
        #________________________________________________________________________________________________________________________
        #Layer 2 
        self.layer2_conv=torch.nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1)
        self.layer2_avgpool= torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
        self.layer2_bn=torch.nn.BatchNorm2d(64, affine=True)
        #________________________________________________________________________________________________________________________
        #Layer 3
        self.layer3_avgpool= torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
        self.layer3_conv=torch.nn.Conv2d(64, 16, kernel_size=3, stride=1, padding=1)
        self.layer3_bn=torch.nn.BatchNorm2d(16, affine=True)
        #________________________________________________________________________________________________________________________
        #Layer 4
        self.layer4_conv=torch.nn.Conv2d(16, 16, kernel_size=3, stride=1, padding=1)
        self.layer4_avgpool= torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
        self.layer4_bn=torch.nn.BatchNorm2d(16, affine=True)
        #________________________________________________________________________________________________________________________
        #Layer 5
        self.layer5_avgpool= torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
        self.layer5_conv=torch.nn.Conv2d(16, 22, kernel_size=3, stride=1, padding=1)
        self.layer5_bn=torch.nn.BatchNorm2d(22, affine=True)
        #________________________________________________________________________________________________________________________   
        self.pool = torch.nn.AdaptiveAvgPool2d((3, 3))
        self.fc1 = torch.nn.Linear(198, 32) 
        self.fc2 = torch.nn.Linear(32, 32) 
        self.fc3 = torch.nn.Linear(32, 32)  
        self.relu = torch.nn.ReLU(inplace=False)
        self.classifier = torch.nn.Linear(32, 10)

    def forward(self, x):
        #________________________________________________________________________________________________________________________
        x = self.layer0_conv(x)
        X = self.layer0_bn(x)
        x= self.relu(x)
        #________________________________________________________________________________________________________________________
        # Unroll layer1
        x = self.layer1_avgpool(x)
        x = self.layer1_conv(x)
        x = self.layer1_bn(x)
        x= self.relu(x)
        #print(f'After l1: {x.shape}')
        #________________________________________________________________________________________________________________________
        # Unroll layer2
        x = self.layer2_conv(x)
        x = self.layer2_avgpool(x)
        x = self.layer2_bn(x)
        x= self.relu(x)
        #print(f'After l2: {x.shape}')
        #________________________________________________________________________________________________________________________
        # Unroll layer3
        x = self.layer3_avgpool(x)
        x = self.layer3_conv(x)
        x = self.layer3_bn(x)
        x= self.relu(x)
        #print(f'After l3: {x.shape}')
        #________________________________________________________________________________________________________________________
        # First AvgPool after layer3
        x = torch.nn.AvgPool2d(kernel_size=2, stride=2)(x)
        #print(f'After intermadiate pool 1: {x.shape}')
        #________________________________________________________________________________________________________________________
        # Unroll layer4
        x = self.layer4_conv(x)
        x = self.layer4_avgpool(x)
        x = self.layer4_bn(x)
        x= self.relu(x)
        #print(f'After l4: {x.shape}')      
        #________________________________________________________________________________________________________________________
        # Unroll layer5
        x = self.layer5_avgpool(x)
        x = self.layer5_conv(x)
        x = self.layer5_bn(x)
        x= self.relu(x)
        #print(f'After l5: {x.shape}')     
        #________________________________________________________________________________________________________________________
        # second AvgPool after layer5
        x = torch.nn.AvgPool2d(kernel_size=2, stride=2)(x)
        #print(f'After intermediate pool 2 {x.shape}')
        #________________________________________________________________________________________________________________________
        x =  self.pool(x)
        #print(f'After adaptive: {x.shape}')
        #________________________________________________________________________________________________________________________
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x= self.relu(x)
        x = self.fc2(x)
        x= self.relu(x)
        x = self.fc3(x)
        x= self.relu(x)
        
        x = self.classifier(x)
        return x

    def set_drop_path_prob(self, drop_path_prob):
        self.drop_path_prob = drop_path_prob

# Data Loaders

In [43]:

def get_cifar10_dataset(train: bool = True, cutout: bool = False):
    if train:
        transform_list = [
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ToTensor(), 
        ]
        transform = transforms.Compose(transform_list)
    else:
        transform = transforms.Compose([
            transforms.ToTensor(), 
        ])


    dataset = nni.trace(datasets.CIFAR10)(root='./data', train=train, download=True, transform=transform)
    
    return dataset


batch_size = 128
train_data = get_cifar10_dataset(train=True)
test_data = get_cifar10_dataset(train=False)  

trainloader = DataLoader(
    train_data, batch_size=batch_size,
    shuffle=True,
    pin_memory=True, num_workers=6, persistent_workers=True
)

testloader = DataLoader(
    test_data, batch_size=256,
    pin_memory=True, num_workers=6, persistent_workers=True
)

Files already downloaded and verified
Files already downloaded and verified


# Training

In [ ]:
model = PhotonicArch().to(DEVICE)
model.load_state_dict(torch.load("best_model4.pth", map_location=DEVICE))

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(
    model.parameters(), 
    lr=LEARNING_RATE, 
    momentum=0.9, 
    weight_decay=0.
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=600  
)
best_acc = 80.97
def train(model, trainloader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, targets in trainloader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
    
    train_acc = 100. * correct / total
    print(f"Train Loss: {running_loss/len(trainloader):.4f} | Train Acc: {train_acc:.2f}%")
    return train_acc

def test(model, testloader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in testloader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    
    test_acc = 100. * correct / total
    print(f"Test Loss: {running_loss/len(testloader):.4f} | Test Acc: {test_acc:.2f}%")
    return test_acc

for epoch in range(1, EPOCHS + 1):
    print(f"Epoch [{epoch}/{EPOCHS}] Current batch_size {batch_size} - best test acc {best_acc}")
    if epoch % 50 == 0 and batch_size > 2:
        batch_size = int(batch_size / 2)  
        print(f"batchsize reduced to {batch_size}")
        trainloader = DataLoader(
            train_data, 
            batch_size=batch_size,
            pin_memory=True,
            num_workers=6,
            persistent_workers=True,
            shuffle=True
        )
    train(model, trainloader, optimizer, criterion)
    test_acc = test(model, testloader, criterion)
    scheduler.step()
    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(), "best_model4.pth")
        print(f"Best model saved with Test Acc: {best_acc:.2f}%")


Epoch [1/600] Current batch_size 128 - best test acc 80.97
